<a href="https://colab.research.google.com/github/ommestriker007/Flood-Mapping/blob/main/Flood_Extent_Mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import geemap
import ee
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd


In [ ]:
ee.Authenticate()
ee.Initialize(project = 'neon-feat-313910')

In [ ]:
aoi_ee = ee.Geometry.Rectangle([74.4, 16.8, 74.7, 17.05])

# before_start = '2021-06-01'
# before_end   = '2021-07-01'   # Pre-flood (dry period)

# after_start  = '2021-07-25'
# after_end    = '2021-08-20'   # During/post-flood


In [ ]:
# ---- 2. Dates (adjust to your flood event) ----
before_start = '2021-06-01'
before_end   = '2021-07-01'
after_start  = '2021-07-25'
after_end    = '2021-08-20'

# ---- 3. Load Sentinel-1 ----
s1_collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi_ee)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    .select('VV')
)

before_collection = s1_collection.filterDate(before_start, before_end)
after_collection  = s1_collection.filterDate(after_start,  after_end)

# ---- 4. Speckle Filter ----
def apply_speckle_filter(image):
    return image.focal_mean(
        radius=50,
        kernelType='circle',
        units='meters'
    ).copyProperties(image, ['system:time_start'])

before_filtered = before_collection.map(apply_speckle_filter).mean().clip(aoi_ee)
after_filtered  = after_collection.map(apply_speckle_filter).mean().clip(aoi_ee)

# ---- 5. Change Detection ----
flood_change = before_filtered.subtract(after_filtered).rename('flood_change')
print("✅ flood_change created successfully")

# ---- 6. Otsu Thresholding ----
def otsu_threshold(image, aoi, scale=90):
    histogram = image.reduceRegion(
        reducer=ee.Reducer.histogram(255, 0.1),
        geometry=aoi,
        scale=scale,
        bestEffort=True
    ).get('flood_change')

    counts  = ee.Array(ee.Dictionary(histogram).get('histogram'))
    means   = ee.Array(ee.Dictionary(histogram).get('bucketMeans'))

    size    = means.length().get([0])
    total   = counts.reduce(ee.Reducer.sum(), [0]).get([0])
    sum_val = means.multiply(counts).reduce(ee.Reducer.sum(), [0]).get([0])
    mean    = sum_val.divide(total)

    indices = ee.List.sequence(1, size)

    def compute_bsv(i):
        aCounts = counts.slice(0, 0, i)
        aCount  = aCounts.reduce(ee.Reducer.sum(), [0]).get([0])
        aMeans  = means.slice(0, 0, i)
        aSum    = aMeans.multiply(aCounts).reduce(ee.Reducer.sum(), [0]).get([0])
        aMean   = aSum.divide(aCount)
        bCount  = total.subtract(aCount)
        bMean   = sum_val.subtract(aSum).divide(bCount)
        return aCount.multiply(aMean.subtract(mean).pow(2)).add(
               bCount.multiply(bMean.subtract(mean).pow(2)))

    bsvs     = indices.map(compute_bsv)
    maxBSV   = bsvs.reduce(ee.Reducer.max())
    maxIndex = bsvs.indexOf(maxBSV)
    return means.get([maxIndex])

auto_threshold  = otsu_threshold(flood_change, aoi_ee)
threshold_val   = auto_threshold.getInfo()
print(f"✅ Otsu Threshold: {threshold_val:.3f} dB")

# ---- 7. Apply Threshold ----
flood_mask_otsu = flood_change.gt(ee.Image.constant(auto_threshold))
print("✅ flood_mask_otsu ready — proceed to Step 6 (Refinement)")


✅ flood_change created successfully
✅ Otsu Threshold: 1.100 dB
✅ flood_mask_otsu ready — proceed to Step 6 (Refinement)


In [ ]:
# --- Remove isolated pixels (salt & pepper) ---
connections = flood_mask_otsu.connectedPixelCount(25, True)
flood_clean = flood_mask_otsu.updateMask(connections.gte(8))

# --- Remove permanent water bodies using JRC water layer ---
jrc_water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('seasonality')
permanent_water = jrc_water.gte(10)           # water present 10+ months/yr
flood_clean = flood_clean.where(permanent_water, 0)

# --- Remove steep slopes (can't flood steep terrain) ---
dem   = ee.Image('WWF/HydroSHEDS/03VFDEM')
slope = ee.Algorithms.Terrain(dem).select('slope')
flood_clean = flood_clean.updateMask(slope.lt(5))

# Final flood mask
flood_extent = flood_clean.updateMask(flood_clean)
print("Flood mask refined!")


Flood mask refined!


In [ ]:
# Calculate flooded area in sq. km
flood_area_stats = flood_extent.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi_ee,
    scale=10,
    bestEffort=True
)

flood_area_km2 = (ee.Number(flood_area_stats.get('flood_change'))
                    .divide(1e6).round())

print(f"Total Flooded Area: {flood_area_km2.getInfo()} km²")


Total Flooded Area: 360 km²


In [ ]:
Map = geemap.Map()
Map.centerObject(aoi_ee, 11)

# Before & After SAR
Map.addLayer(before_filtered, {'min': -25, 'max': 0}, 'SAR Before (VV)')
Map.addLayer(after_filtered,  {'min': -25, 'max': 0}, 'SAR After (VV)')

# Flood Change layer (heatmap style)
Map.addLayer(flood_change,
             {'min': 0, 'max': 2, 'palette': ['white', 'cyan', 'blue']},
             'Flood Change (dB)')

# Final Flood Extent (bright red)
Map.addLayer(flood_extent,
             {'min': 0, 'max': 1, 'palette': ['#FF0000']},
             'Flood Extent')

# AOI boundary
Map.addLayer(aoi_ee, {'color': 'yellow'}, 'AOI Boundary')

Map.addLayerControl()
Map


Map(center=[16.925027023306654, 74.55000000000061], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Export flood map as GeoTIFF to Google Drive
task = ee.batch.Export.image.toDrive(
    image=flood_extent,
    description='flood_extent_map',
    folder='FloodMapping_Project',
    fileNamePrefix='flood_extent',
    region=aoi_ee,
    scale=10,
    crs='EPSG:4326',
    maxPixels=1e10
)
task.start()
print("Export started! Check Google Drive > FloodMapping_Project folder.")

# Export as Shapefile vector too
flood_vectors = flood_extent.reduceToVectors(
    geometry=aoi_ee,
    scale=10,
    geometryType='polygon',
    eightConnected=True,
    bestEffort=True
)

vector_task = ee.batch.Export.table.toDrive(
    collection=flood_vectors,
    description='flood_extent_vector',
    folder='FloodMapping_Project',
    fileFormat='SHP'
)
vector_task.start()
print("Vector export started!")


Export started! Check Google Drive > FloodMapping_Project folder.
Vector export started!
